In [ ]:
%pip install -q "math-verify[antlr4_13_2]==0.9.0" torch \
  "transformers==5.0.0" "trl==1.10.0" "peft==0.19.1" \
  accelerate datasets "torchao>=0.16.0"

import subprocess, os, sys, base64, io, tarfile
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "/kaggle/working/DACNTT_2026"])
if subprocess.run(["git", "clone", "https://github.com/tsan2711/DACNTT_2026.git", "/kaggle/working/DACNTT_2026"]).returncode != 0:
    raise SystemExit("git clone thất bại")
os.chdir("/kaggle/working/DACNTT_2026")
BLOB = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(BLOB)), mode="r:gz").extractall(".")
env = os.environ.copy()
env["PYTHONPATH"] = "src:."
env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
heartbeat = subprocess.Popen(
    [sys.executable, "-u", "experiments/viec3/heartbeat.py", "/kaggle/working/heartbeat.log"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, stdin=subprocess.DEVNULL, start_new_session=True)
print("setup xong")


In [ ]:
import subprocess, signal, os
out = "/kaggle/working/results/viec3/gate-teacher-s2"
cmd = ["python", "-u", "experiments/viec3/kaggle_driver.py", "--out", out, "--rounds", "2",
       "--log", "/kaggle/working/run_gate-teacher-s2.log", "--idle-minutes", "25", "--budget-hours", "5", "--",
       "python", "-u", "-m", "experiments.viec3.run", "--mode", "hf", "--model", "Qwen/Qwen2.5-1.5B-Instruct", "--dataset", "math500",
       "--n", "500", "--rounds", "2", "--k", "2", "--max-tokens", "1024", "--seed", "2", "--learning-rate", "1e-4", "--fixed-pipeline", "--train-pool-n", "300", "--completion-only", "--lora-targets", "all-linear", "--max-seq-length", "1024", "--train-on-gold", "--teacher-solutions", "experiments/viec3/teacher_solutions.jsonl", "--max-train-examples", "298", "--out", out]
p = subprocess.Popen(cmd, env=env, start_new_session=True)
try:
    code = p.wait(timeout=5.5 * 3600)
except subprocess.TimeoutExpired:
    os.killpg(p.pid, signal.SIGKILL); code = "TIMEOUT"
print("gate-teacher-s2 driver exit:", code)
subprocess.run(["python", "experiments/viec3/review_cpu/gate_anatomy.py", out])
